# 第 7 章 机器人操作系统的应用


## 输入模块


图 7-12 为输入模块 Simulink 框图。

{: .figure-caption}



![](assets/e396cf73922071592846441d50df2971b6306a161bed5b8b8823336d712bafbf.jpg)

图 7-12 输入模块 Simulink 框图

{: .figure-caption}

Decoder1 的相关代码为：



In [ ]:
function [left_lamda, left_qd, left_dqd, left_q, left_dq, right_lamda, right_qd, right_dqd,
right_q, right_dq] = fcn(data, N)
persistent Data;
if isempty(Data)
    Data = single(data);
end
left_lamda = [0 0 0 0 0 0 0]';
left_qd = [0 0 0 0 0 0 0]';
left_dqd = [0 0 0 0 0 0 0]';
left_q = [0 0 0 0 0 0 0]';
left_dq = [0 0 0 0 0 0 0]';
right_lamda = [0 0 0 0 0 0 0]';
right_qd = [0 0 0 0 0 0 0]';
right_dqd = [0 0 0 0 0 0 0]';
right_q = [0 0 0 0 0 0 0]';
right_dq = [0 0 0 0 0 0 0]';
if N ~= 0
    Data = single(data);
end

for i = 0:6
    left_lamda(i+1) = (Data(1+2*i) + Data(2+2*i) *256)/1000.0;
    left_qd(i+1) = (Data(15+2*i) + Data(16+2*i) *256 - 32768)/10000.0;
    left_dqd(i+1) = (Data(29+2*i) + Data(30+2*i) *256 - 32768)/10000.0;
    left_q(i+1) = (Data(43+2*i) + Data(44+2*i) *256 - 32768)/10000.0;
    left_dq(i+1) = (Data(57+2*i) + Data(58+2*i) *256 - 32768)/10000.0;

    right_lamda(i+1) = (Data(71+2*i) + Data(72+2*i) *256)/1000.0;
    right_qd(i+1) = (Data(85+2*i) + Data(86+2*i) *256 - 32768)/10000.0;
    right_dqd(i+1) = (Data(99+2*i) + Data(100+2*i) *256 - 32768)/10000.0;
    right_q(i+1) = (Data(113+2*i) + Data(114+2*i) *256 - 32768)/10000.0;
    right_dq(i+1) = (Data(127+2*i) + Data(128+2*i) *256 - 32768)/10000.0;
end


## 数据处理模块


图 7-13 为数据处理模块 Simulink 框图。

{: .figure-caption}



![](assets/73b237b62d2167df3366722a22d937b6a5906ad511a11485bba5734cf9d23d8d.jpg)

图 7-13 数据处理模块 Simulink 框图

{: .figure-caption}

其中，matricsZ\_g 的代码为：



In [ ]:
function Z_g = matricesZ_g(q)
n = 3;
persistent qlmt;
persistent qn;
if isempty(qlmt)
    qlmt = [-1.7 1.7; -2.14 1.05; -3.05 3.05; -0.05 2.61; -3.05 3.05; -1.57 2.09; -3.05 3.05];
    qn = zeros(7, n);
    for i = 1:7
    for j = 1:n
    qn(i, j) = qlmt(i, 1) + (qlmt(i, 2) - qlmt(i, 1)) / (n - 1) * (j - 1);
    end
    end
end
Zraw = zeros(1, n^7);
counter = 1;
for i1 = 1:n
    for i2 = 1:n
    for i3 = 1:n
    for i4 = 1:n
    for i5 = 1:n
    for i6 = 1:n
    for i7 = 1:n
c = [qn(1, i1), qn(2, i2), qn(3, i3), qn(4, i4), qn(5, i5), qn(6, i6), qn(7, i7)]';
    Zraw(1, counter) = RBF(q, c);
    counter = counter + 1;
    end
    end
    end
    end
    end
    end
end
end
Z_g = [Zraw Zraw Zraw Zraw Zraw Zraw Zraw]';
function out = RBF(x, c)
out = exp(-(x - c)'*(x - c)/0.5));


## 输出模块


图 7-14 为输出模块 Simulink 框图。

{: .figure-caption}



![](assets/88aadfaf7ef638f0d8d25f8cc825e4db37dd6d5cc42c7270bb9a2a0c5e00c75d.jpg)

图 7-14 输出模块 Simulink 框图

{: .figure-caption}

Encoder 模块代码如下：



In [ ]:
function data = fcn(tau1_left, tau1_right)
data = uint8([0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]);
dataraw = tau1_left;
dataraw = uint16(dataraw * 1000 + 32768);
data(1) = uint8(bitand(dataraw(1), hex2dec('FF00')) / 256);
data(2) = uint8(dataraw(1) - uint16(data(1)) * 256);
data(3) = uint8(bitand(dataraw(2), hex2dec('FF00')) / 256);
data(4) = uint8(dataraw(2) - uint16(data(3)) * 256);
data(5) = uint8(bitand(dataraw(3), hex2dec('FF00')) / 256);
data(6) = uint8(dataraw(3) - uint16(data(5)) * 256);
data(7) = uint8(bitand(dataraw(4), hex2dec('FF00')) / 256);
data(8) = uint8(dataraw(4) - uint16(data(7)) * 256);
data(9) = uint8(bitand(dataraw(5), hex2dec('FF00')) / 256);
data(10) = uint8(dataraw(5) - uint16(data(9)) * 256);
data(11) = uint8(bitand(dataraw(6), hex2dec('FF00')) / 256);
data(12) = uint8(dataraw(6) - uint16(data(11)) * 256);
data(13) = uint8(bitand(dataraw(7), hex2dec('FF00')) / 256);
data(14) = uint8(dataraw(7) - uint16(data(13)) * 256);

dataraw = tau1_right;
dataraw = uint16(dataraw * 1000 + 32768);
data(15) = uint8(bitand(dataraw(1), hex2dec('FF00')) / 256);
data(16) = uint8(dataraw(1) - uint16(data(15)) * 256);
data(17) = uint8(bitand(dataraw(2), hex2dec('FF00')) / 256);
data(18) = uint8(dataraw(2) - uint16(data(17)) * 256);
data(19) = uint8(bitand(dataraw(3), hex2dec('FF00')) / 256);
data(20) = uint8(dataraw(3) - uint16(data(19)) * 256);
data(21) = uint8(bitand(dataraw(4), hex2dec('FF00')) / 256);
data(22) = uint8(dataraw(4) - uint16(data(21)) * 256);
data(23) = uint8(bitand(dataraw(5), hex2dec('FF00')) / 256);
data(24) = uint8(dataraw(5) - uint16(data(23)) * 256);
data(25) = uint8(bitand(dataraw(6), hex2dec('FF00')) / 256);
data(26) = uint8(dataraw(6) - uint16(data(25)) * 256);
data(27) = uint8(bitand(dataraw(7), hex2dec('FF00')) / 256);
data(28) = uint8(dataraw(7) - uint16(data(27)) * 256);


## Python 代码


以下关节空间控制器和 PID 控制器的定义有所不同，程序代码如下:



In [ ]:
class CJointController:
    def __init__(self, ArmName):


In [ ]:
self.name = ArmName
self.qd = np.mat([[0.0], [0.0], [0.0], [0.0], [0.0], [0.0]])
self.dotqd = np.mat([[0.0], [0.0], [0.0], [0.0], [0.0], [0.0]])
# self.qcmd = dict()
self.limb = baxter_interface.Limb(ArmName)
self.limb.set_joint_position_speed(1.0)self.limb.set_joint_position_speed(0.5)
self.torController0 = CPIDController(self.name + 'j0')CPIDController()
self.torController1 = CPIDController(self.name + 'j1')CPIDController()
self.torController2 = CPIDController(self.name + 'j2')CPIDController()
self.torController3 = CPIDController(self.name + 'j3')CPIDController()
self.torController4 = CPIDController(self.name + 'j4')CPIDController()
self.torController5 = CPIDController(self.name + 'j5')CPIDController()
self.torController6 = CPIDController(self.name + 'j6')CPIDController()
self.torController = [self.torController0, self.torController1,
self.torController2, self.torController3, self.torController4, self.torController5,
self.torController6]

self.krate = 1.0

self.torController[0].kp = 60.0
self.torController[0].kd = 6.0#21.0
# self.torController[0].ki = 0.01

self.torController[1].kp = 60
self.torController[1].kd = 6.0#24.0
# self.torController[1].ki = 0.01

self.torController[2].kp = 50.0
self.torController[2].kd = 5.0#18.0
# self.torController[2].ki = 0.01

self.torController[3].kp = 30.0
self.torController[3].kd = 3.0#15.0
# self.torController[3].ki = 0.01

self.torController[4].kp = 11.0
self.torController[4].kd = 1.1#5.4
# self.torController[4].ki = 0.01

self.torController[5].kp = 8.0
self.torController[5].kd = 0.8#4.5
# self.torController[5].ki = 0.01

self.torController[6].kp = 5.0
self.torController[6].kd = 0.5#7.5
# self.torController[6].ki = 0.002


In [ ]:
# # # # # # # # # # # # # # # class CPIDController:
    def __init__(self, name):__init__(self):
    self.kp = 1.0
    self.ki = 0.0
    self.kd = 0.0
    self.sum_max = 1000.0
    self.sum_min = -1000.0
    self.err_old = 0.0
    self.err_sum = 0.0
    self.derr_old = 0.0
    self.sumrate = 1.0
    self.rho = 0.1
    self.name = name

    timestr = time.strftime("%M%H-%d%m")
    self.file = open(timestr + self.name + ".csv", 'w')
    tempstr = "time,given,dotgiven,feedback,dotfeedback,err,derr,errtr,out\r\n"
    self.file.write(tempstr)

    def calculate_dot(self, given, dotgiven, feedback, dotfeedback):
    err = given - feedback
    derr = dotgiven - dotfeedback
    self.derr_old = (self.derr_old * 1.0 + derr) / 2.0
    self.err_old = given
    self.err_sum += err
    if self.err_sum > self.sum_max * self.sumrate:
    self.err_sum = self.sum_max * self.sumrate
    elif self.err_sum    self.sum_min * self.sumrate:
    self.err_sum = self.sum_min * self.sumrate
    #out = self.kp * err + self.kd * derr + self.ki * self.err_sum
    out = self.kp * err + self.kd * self.derr_old + self.ki * self.err_sum
    tempstr = str(rospy.get_time()) + ", " + str(given) + ", " + str(dotgiven) + ", " + str(feedback) + ", " + str(dotfeedback) + ", " + str(err) + ", " + str(derr) + ", " + str(err) + ", " + str(out) + ", \r\n"
    self.file.write(tempstr)
    return out

def calculate_dot_tr(self, given, dotgiven, feedback, dotfeedback):
    #tempstr = "time,given,dotgiven,feedback,dotfeedback,err,derr,errtr,out\r\n"
    err = given - feedback


In [ ]:
derr = dotgiven - dotfeedback
self.derr_old = (self.derr_old * 1.0 + derr) / 2.0
self.err_old = given
self.err_sum += err
errtemp = err
if errtemp > 0.99999 * self.rho:
    errtemp = 0.99999 * self.rho
if errtemp - 0.99999 * self.rho:
    errtemp = -0.99999 * self.rho
errtr = 0.5 * self.rho * np.log((errtemp / self.rho + 1.0) / (1.0 - errtemp / self.rho))
if errtr > 0.2:
    errtr = 0.2
if errtr - 0.2:
    errtr = -0.2
if self.err_sum > self.sum_max * self.sumrate:
    self.err_sum = self.sum_max * self.sumrate
elif self.err_sum    self.sum_min * self.sumrate:
    self.err_sum = self.sum_min * self.sumrate
#out = self.kp * err + self.kd * derr + self.ki * self.err_sum
out = self.kp * errtr + self.kd * self.derr_old + self.ki * self.err_sum
tempstr = str(rospy.get_time()) + ", " + str(given) + ", " + str(dotgiven) + ", " + str(feedback) + ", " + str(dotfeedback) + ", " + str(err) + ", " + str(derr) + ", " + str(errtr) + ", " + str(out) + ", \r\n"
self.file.write(tempstr)
return out


以下 3 个 m 文件是共同工作的，它们的作用是生成机器人的轨迹，并转化为机器人末端执行器的轨迹，其中定时器起到了控制时序作用。



## (1) 轨迹设置相关代码如下:


In [ ]:
clear all
close all
global t;
t = 0;
us = udp('192.168.1.100', 6101);
fopen(us);
ti = timer();
set(ti, 'BusyMode', 'drop', 'Period', 0.1, 'ExecutionMode', 'fixedRate')
set(ti, 'TimerFcn', {@timeChange, us})
start(ti);
t


(2) 轨迹空间转换代码如下:



In [ ]:
function baxterPos = moveBaxHand_twoarms(us, leftpos, rightpos, handName)
% 移动 baxter 的手臂到坐标 baxxyz。
% baxxyz 为在 baxter 坐标系中期望位置坐标, handName 为想要移动的手臂名称, 分别为 left, right 和 both
lftrpy = [pi 0 pi]';
rgtrpy = [pi 0 pi]';
% lftrpy = [-pi/2 0 0.1]';
% rgtrpy = [-pi/2 0 0]';
a1 = size(leftpos);
a = size(rightpos);

if a1(1) == 1 & a1(2) == 3 % 自动调整坐标为列向量, 因为下文中的 encoder 要求为列向量
    leftpos = leftpos';
elseif a1(1) == 3 & a1(2) == 1
    leftpos;
else
    display('baxxyz error')
    baxterPos = [0 0 0];
    return;
end

if a(1) == 1 & a(2) == 3 % 自动调整坐标为列向量, 因为下文中的 encoder 要求为列向量
    rightpos = rightpos';
elseif a(1) == 3 & a(2) == 1
    rightpos;
else
    display('baxxyz error')
    baxterPos = [0 0 0];
    return;
end

if isequal(handName, 'right') % 移动左手或右手或判断移动哪只手
    [data] = encoder1([0.5805 0.2625 0.2141]', rightpos, lftrpy, rgtrpy, [255 255]);
    fwrite(us, data);
elseif isequal(handName, 'left')
    [data] = encoder1(leftpos, [0.5735 - 0.224 0.2141]', lftrpy, rgtrpy, [255 255]);
    fwrite(us, data);
elseif isequal(handName, 'both')
    % [yz j] = theloop2(baxxyz);
    [data] = encoder1(leftpos, rightpos, lftrpy, rgtrpy, [0 255]);
    fwrite(us, data);
end
baxterPos = [0 0 0];

end


## (3) 定时器相关代码如下:


In [ ]:
function timeChange(obj, event, us)
% UNTITLED2 Summary of this function goes here
% Detailed explanation goes here
global t;
t = t + obj.Period;
length = 0.1;
tstep = 2.5;
r1 = 0.12;
r2 = 0.1;

xstartl = 0.65;
xstartr = 0.65;
ystartl = 0.2;
ystartr = -0.2;

% % circular trajectory
% experiment 1
leftpos = [r1 * sin(t * 1 * pi/tstep) + xstartl, r2 * cos(t * 1 * pi/tstep) + ystartl, 0.2];
rightpos = [r1 * sin(t * 1 * pi/tstep) + xstartr, r2 * cos(t * 1 * pi/tstep) + ystartr, 0.2];

% % set point
moveBaxHand_twoarms_3joints(us, leftpos, rightpos, 'both');

t
end


## Simulink 模块


Decoder1 模块和 Decoder2 模块的作用是将 UDP 传输的数据转化为机器人左臂和右臂的关节角、角速度、角加速度、控制增益等，然后在通过其他 Simulink 模块实现设计的控制算法。

(1) Decoder1 模块的 m 代码如下:



In [ ]:
function [left_lamda, left_qd, left_dqd, left_q, left_dq, right_lamda, right_qd, right_dqd, right_q, right_dq, left_torque, right_torque, left_exforce, right_exforce] = fcn(data, N)
persistent Data;
if isempty(Data)
    Data = single(data);
end
left_lamda = [0 0 0 0 0 0 0]';
left_qd = [0 0 0 0 0 0 0]';
left_dqd = [0 0 0 0 0 0 0]';
left_q = [0 0 0 0 0 0 0]';
left_dq = [0 0 0 0 0 0 0]';


In [ ]:
right_lamda = [0 0 0 0 0 0 0]';
right_qd = [0 0 0 0 0 0 0]';
right_dqd = [0 0 0 0 0 0 0]';
right_q = [0 0 0 0 0 0 0]';
right_dq = [0 0 0 0 0 0 0]';
left_torque = [0 0 0 0 0 0 0]';
right_torque = [0 0 0 0 0 0 0]';
left_exforce = [0 0 0 0 0 0]';
right_exforce = [0 0 0 0 0 0]';
if N ~= 0
    Data = single(data);
end

for i = 0:6
    left_lamda(i + 1) = (Data(1 + 2 * i) + Data(2 + 2 * i) * 256)/1000.0;
    left_qd(i + 1) = (Data(15 + 2 * i) + Data(16 + 2 * i) * 256 - 32768)/10000.0;
    left_dqd(i + 1) = (Data(29 + 2 * i) + Data(30 + 2 * i) * 256 - 32768)/10000.0;
    left_q(i + 1) = (Data(43 + 2 * i) + Data(44 + 2 * i) * 256 - 32768)/10000.0;
    left_dq(i + 1) = (Data(57 + 2 * i) + Data(58 + 2 * i) * 256 - 32768)/10000.0;
    right_lamda(i + 1) = (Data(71 + 2 * i) + Data(72 + 2 * i) * 256)/1000.0;
    right_qd(i + 1) = (Data(85 + 2 * i) + Data(86 + 2 * i) * 256 - 32768)/10000.0;
    right_dqd(i + 1) = (Data(99 + 2 * i) + Data(100 + 2 * i) * 256 - 32768)/10000.0;
    right_q(i + 1) = (Data(113 + 2 * i) + Data(114 + 2 * i) * 256 - 32768)/10000.0;
    right_dq(i + 1) = (Data(127 + 2 * i) + Data(128 + 2 * i) * 256 - 32768)/10000.0;
end
for i = 0:6
    left_torque(i + 1) = (Data(141 + 2 * i) + Data(142 + 2 * i) * 256 - 32768)/1000.0;
    right_torque(i + 1) = (Data(155 + 2 * i) + Data(156 + 2 * i) * 256 - 32768)/1000.0;
end
for i = 0:5
    left_exforce(i + 1) = (Data(169 + 2 * i) + Data(170 + 2 * i) * 256 - 32768)/1000.0;
    right_exforce(i + 1) = (Data(181 + 2 * i) + Data(182 + 2 * i) * 256 - 32768)/1000.0;
end


## (2) Decoder2 模块的 m 代码如下:


In [ ]:
function [leftpos, rightpos, leftrpy, rightrpy, grippers, leftposd, rightposd, leftxyzd, rightxyzd] = fcn(data,N)
persistent Data;
if isempty(Data)
    Data = single(data);
end
leftpos = [0 0 0]';
rightpos = [0 0 0]';


In [ ]:
leftposd = [0 0 0]';
rightposd = [0 0 0]';
leftrpy = [0 0 0]';
rightprpy = [0 0 0]';
grippers = [0 0]';
leftxyzd = [0 0 0]';
rightxyzd = [0 0 0]';
Data = single(data);
for i = 0:2
    leftrpy(i + 1) = (Data(7 + 2 * i) + Data(8 + 2 * i) * 256 - 32768)/10000.0;
    leftpos(i + 1) = (Data(13 + 2 * i) + Data(14 + 2 * i) * 256 - 32768)/10000.0;
    rightprpy(i + 1) = (Data(25 + 2 * i) + Data(26 + 2 * i) * 256 - 32768)/10000.0;
    rightpos(i + 1) = (Data(31 + 2 * i) + Data(32 + 2 * i) * 256 - 32768)/10000.0;
    leftposd(i + 1) = (Data(45 + 2 * i) + Data(46 + 2 * i) * 256 - 32768)/10000.0;
    rightposd(i + 1) = (Data(51 + 2 * i) + Data(52 + 2 * i) * 256 - 32768)/10000.0;
    leftxyzd(i + 1) = (Data(57 + 2 * i) + Data(58 + 2 * i) * 256 - 32768)/10000.0;
    rightxyzd(i + 1) = (Data(63 + 2 * i) + Data(64 + 2 * i) * 256 - 32768)/10000.0;
end

grippers(1) = Data(37);
grippers(1) = Data(38);


本节内容所设计的框架如图 7-41 所示，其中 C++ 程序用于接收 Kinect 传感器传输的数据，并进行处理，生成关节角度 $q_{d}$ 。Python 程序主要是围绕主控制器而设计的，对关节角度进行处理后输出力矩 $\tau$ , 而 MATLAB/Simulink 运用神经网络控制算法，系统的未知项进行估计，相当于前馈控制，并生成了补偿力矩 $\tau_{c}$ 。



![](assets/d680cfb7bfbc2754e0b24e64f03d4d3aa3e3c8be121eb1b47d2b685e5a36032a.jpg)

图 7-41 实验程序框架

{: .figure-caption}



## Python 代码


该 Python 程序主要实现的功能是接收并处理 Kinect 发送的相关数据。它将接收到的字符串型数据改为浮点型数据，并将人体的关节空间映射到机器人运动的关节空间，已达到使用 Kinect 摇操作机器人的目的，程序代码如下：



In [ ]:
class CudptoFFAT:
    def __init__(self, leftarm, rightarm):
    self.HOST = '192.168.1.100'
    self.PORT = 6103
    self.BUFSIZE = 1024
    self.ADDR = (self.HOST, self.PORT)
    self.RemotePORT = 6103
    self.udpSock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    self.udpSock.settimeout(1)
    self.udpSock.bind(self.ADDR)
    self.connected = False
    self.t = threading.Thread(target=self.listenWorker)
    self.thread_stop = False
    self.leftarm = leftarm
    self.rightarm = rightarm
    self.PI = 3.14159267
    self.qd = np.mat([[0.0], [0.0], [0.0], [0.0], [0.0], [0.0]])
    self.errcounter = 0
    self.t.start()
def Decode(self, data):
    self.FATResultLeft[0, 0] = ((ord(data[0]) * 256 + ord(data[1])) - 32768.0) / 1000.0
    self.FATResultLeft[1, 0] = ((ord(data[2]) * 256 + ord(data[3])) - 32768.0) / 1000.0


In [ ]:
self.FATResultLeft[2,0] = ((ord(data[4]) * 256 + ord(data[5])) - 32768.0)/1000.0
self.FATResultLeft[3,0] = ((ord(data[6]) * 256 + ord(data[7])) - 32768.0)/1000.0
self.FATResultLeft[4,0] = ((ord(data[8]) * 256 + ord(data[9])) - 32768.0)/1000.0
self.FATResultLeft[5,0] = ((ord(data[10]) * 256 + ord(data[11])) - 32768.0)/1000.0
self.FATResultLeft[6,0] = ((ord(data[12]) * 256 + ord(data[13])) - 32768.0)/1000.0

self.FATResultRight[0,0] = ((ord(data[14]) * 256 + ord(data[15])) - 32768.0)/1000.0
self.FATResultRight[1,0] = ((ord(data[16]) * 256 + ord(data[17])) - 32768.0)/1000.0
self.FATResultRight[2,0] = ((ord(data[18]) * 256 + ord(data[19])) - 32768.0)/1000.0
self.FATResultRight[3,0] = ((ord(data[20]) * 256 + ord(data[21])) - 32768.0)/1000.0
self.FATResultRight[4,0] = ((ord(data[22]) * 256 + ord(data[23])) - 32768.0)/1000.0
self.FATResultRight[5,0] = ((ord(data[24]) * 256 + ord(data[25])) - 32768.0)/1000.0
self.FATResultRight[6,0] = ((ord(data[26]) * 256 + ord(data[27])) - 32768.0)/1000.0

def listenWorker(self):
    udp_rate = rospy.Rate(1000.0)
    while not self.thread_stop:
    try:
    data, addr = self.udpSock.recvfrom(self.BUFSIZE)
    self.connected = True
    # print data
    # self декode(data)
    qd = np.mat([ [ 0 . 0 ], [ 0 . 0 ], [ 0 . 0 ], [ 0 . 0 ], [ 0 . 0 ], [ 0 . 0 ]])
    qd[ 0 , 0 ] = float(data[ 0 : 8])
    qd[ 1 , 0 ] = float(data[ 9 : 17])
    qd[ 2 , 0 ] = float(data[ 18 : 26])
    qd[ 3 , 0 ] = float(data[ 27 : 35])
    qd[ 4 , 0 ] = float(data[ 36 : 44])
    qd[ 5 , 0 ] = float(data[ 45 : 53])
    qd[ 6 , 0 ] = float(data[ 54 : 62])

    qd[ 0 , 0 ] = -1.7 + qd[ 0 , 0 ] / self.PI * 1.7
    qd[ 1 , 0 ] = 1.05 - qd[ 1 , 0 ] / self.PI * 3.19 + 0.5
    qd[ 2 , 0 ] = -3.05 + qd[ 2 , 0 ] / self.PI * 3.05
    qd[ 3 , 0 ] = 2.61 - qd[ 3 , 0 ] / self.PI * 2.66
    qd[ 4 , 0 ] = qd[ 4 , 0 ]
    qd[ 5 , 0 ] = qd[ 5 , 0 ]
    qd[ 6 , 0 ] = qd[ 6 , 0 ]


In [ ]:
self.leftarm.qkinect = qd
except:
    self.errcounter += 1
    self.connected = False
    print "FFAT UDP time out. Counter = ", self.errcounter

udp_rate.sleep()

def stop(self):
    self.thread_stop = True
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
###
##
qd = self.qkinect
errqk = self.qkinect - qnow
print 'self.qkinect', self.qkinect, 'errqk = ', errqk
dotqd = self.poscontroller.calculate_mul_err(errqk)
for i in range(0, 7):
    if dotqd[i] > 0.2:
    dotqd[i] = 0.2
    if qnow[i] - 0.2:
    qnow[i] = 0.2

for i in range(0, 7):
    if qnow[i] > self.qmax[i]:
    qnow[i] = self.qmax[i]
    dotqd[i, 0] = 0
    if qnow[i] self.qmin[i]:
    qnow[i] = self.qmin[i]
    dotqd[i, 0] = 0
###


## Kinect 捕捉人体骨骼


(1) 坐标点计算的代码如下:



In [ ]:
HipRight.x += joint[JointType_HipRight].Position.X;
HipRight.y += joint[JointType_HipRight].Position.Y;
HipRight.z += joint[JointType_HipRight].Position.Z;
HipLeft.x += joint[JointType_HipLeft].Position.X;
HipLeft.y += joint[JointType_HipLeft].Position.Y;
HipLeft.z += joint[JointType_HipLeft].Position.Z;
ShoulderRight.x += joint[JointType_ShoulderRight].Position.X;
ShoulderRight.y += joint[JointType_ShoulderRight].Position.Y;


In [ ]:
ShoulderRight.z += joint[JointType_ShoulderRight].Position.Z;
ShoulderLeft.x += joint[JointType_ShoulderLeft].Position.X;
ShoulderLeft.y += joint[JointType_ShoulderLeft].Position.Y;
ShoulderLeft.z += joint[JointType_ShoulderLeft].Position.Z;
ElbowLeft.x += joint[JointType_ElbowLeft].Position.X;
ElbowLeft.y += joint[JointType_ElbowLeft].Position.Y;
ElbowLeft.z += joint[JointType_ElbowLeft].Position.Z;
WristLeft.x += joint[JointType_WristLeft].Position.X;
WristLeft.y += joint[JointType_WristLeft].Position.Y;
WristLeft.z += joint[JointType_WristLeft].Position.Z;
HandLeft.x += joint[JointType_HandLeft].Position.X;
HandLeft.y += joint[JointType_HandLeft].Position.Y;
HandLeft.z += joint[JointType_HandLeft].Position.Z;
ThumbLeft.x += joint[JointType_ThumbLeft].Position.X;
ThumbLeft.y += joint[JointType_ThumbLeft].Position.Y;
ThumbLeft.z += joint[JointType_ThumbLeft].Position.Z;
HandTipLeft.x += joint[JointType_HandTipLeft].Position.X;
HandTipLeft.y += joint[JointType_HandTipLeft].Position.Y;
HandTipLeft.z += joint[JointType_HandTipLeft].Position.Z;
ElbowRight.x += joint[JointType_ElbowRight].Position.X;
ElbowRight.y += joint[JointType_ElbowRight].Position.Y;
ElbowRight.z += joint[JointType_ElbowRight].Position.Z;
WristRight.x += joint[JointType_WristRight].Position.X;
WristRight.y += joint[JointType_WristRight].Position.Y;
WristRight.z += joint[JointType_WristRight].Position.Z;
HandRight.x += joint[JointType_HandRight].Position.X;
HandRight.y += joint[JointType_HandRight].Position.Y;
HandRight.z += joint[JointType_HandRight].Position.Z;
ThumbRight.x += joint[JointType_ThumbRight].Position.X;
ThumbRight.y += joint[JointType_ThumbRight].Position.Y;
ThumbRight.z += joint[JointType_ThumbRight].Position.Z;
HandTipRight.x += joint[JointType_HandTipRight].Position.X;
HandTipRight.y += joint[JointType_HandTipRight].Position.Y;
HandTipRight.z += joint[JointType_HandTipRight].Position.Z;


## (2) 向量和角度计算的代码如下:


In [ ]:
//计算向量；
CvPoint3D32f CO, CD, CB, CE, CH, DC, DD1, DE, EC, ED, EH, EF, EG, L_X5, L_X7;
//LeftArm
CO = GetVector(ShoulderLeft, HipLeft);
CD = GetVector(ShoulderLeft, ElbowLeft);
CB = GetVector(ShoulderLeft, ShoulderRight);
CE = GetVector(ShoulderLeft, WristLeft);
CH = GetVector(ShoulderLeft, ThumbLeft);
DC = GetVector(ElbowLeft, ShoulderLeft);
DD1.x = 0; DD1.y = -1; DD1.z = 0;
DE = GetVector(ElbowLeft, WristLeft);
EC = GetVector(WristLeft, ShoulderLeft);
ED = GetVector(WristLeft, ElbowLeft);
EH = GetVector(WristLeft, ThumbLeft);
EF = GetVector(WristLeft, HandLeft);
EG = GetVector(HandLeft, HandTipLeft);
Left_k = -(EF.x * DE.x + EF.y * DE.y + EF.z * DE.z) / (EH.x * DE.x + EH.y * DE.y + EH.z * DE.z);
L_X5 = VectorPlus(VectorMultiply(Left_k, EH), EF);
L_X7 = CrossProduct(EF, CrossProduct(EF, EH));
//RightArm
CvPoint3D32f BA, BK, BC, BL, KB, KL, LK, LP, LN, LM, R_X5, R_X7;
BA = GetVector(ShoulderRight, HipRight);
BK = GetVector(ShoulderRight, ElbowRight);
BC = GetVector(ShoulderRight, ShoulderLeft);
BL = GetVector(ShoulderRight, WristRight);
KB = GetVector(ElbowRight, ShoulderRight);
KL = GetVector(ElbowRight, WristRight);
LK = GetVector(WristRight, ElbowRight);
LP = GetVector(WristRight, ThumbRight);
LN = GetVector(WristRight, HandTipRight);
LM = GetVector(WristRight, HandRight);
Right_k = -(LM.x * KL.x + LM.y * KL.y + LM.z * KL.z) / (LP.x * KL.x + LP.y * KL.y + LP.z * KL.z);
R_X5 = VectorPlus(VectorMultiply(Right_k, LP), LM);
R_X7 = CrossProduct(LM, CrossProduct(LM, LP));
//计算角度；
/LeftArm
Leftangle1 = PI - AngerOf(CrossProduct(CO, CD), CrossProduct(CB, CO));
Leftangle2 = AngerOf(CO, CD);
Leftangle3 = AngerOf(CrossProduct(CO, CD), CrossProduct(CD, CE));
Leftangle4 = AngerOf(DC, DE);
Leftangle5 = AngerOf(CrossProduct(DE, DD1), CrossProduct(ED, EH));
Leftangle6 = PI / 2 + AngerOf(CrossProduct(EH, EF), ED);


In [ ]:
Leftangle7 = AngerOf(L_X5, L_X7);
//RightArm
Rightangle1 = PI - AngerOf(CrossProduct(BA, BK), CrossProduct(BC, BA));
Rightangle2 = AngerOf(BK, BA);
Rightangle3 = AngerOf(CrossProduct(BA, BK), CrossProduct(BK, BL));
Rightangle4 = AngerOf(KB, KL);
Rightangle5 = AngerOf(CrossProduct(KL, DD1), CrossProduct(LK, LP));
Rightangle6 = PI / 2 + AngerOf(CrossProduct(LP, LM), LK);
Rightangle7 = AngerOf(R_X5, R_X7);


## (3) 画出人体骨骼的代码如下:


In [ ]:
void DrawBone(Mat& SkeletonImage, CvPoint pointSet[], const Joint * pJoints, int whichone, JointType joint0, JointType joint1),
{
    TrackingState joint0State = pJoints[joint0].TrackingState;
    TrackingState joint1State = pJoints[joint1].TrackingState;

    if ((joint0State == TrackingState_NotTracked) || (joint1State == TrackingState_NotTracked))
    {
    return;
    }

    if ((joint0State == TrackingState_Inferred) && (joint1State == TrackingState_Inferred))
    {
    return;
    }

    CvScalar color;
    switch (whichone) //跟踪不同的人显示不同的颜色
    {
    case 0:
    color = cvScalar(255);
    break;
    case 1:
    color = cvScalar(0, 255);
    break;
    case 2:
    color = cvScalar(0, 0, 255);
    break;
    case 3:
    color = cvScalar(255, 255, 0);


In [ ]:
break;
case 4:
    color = cvScalar(255, 0, 255);
    break;
case 5:
    color = cvScalar(0, 255, 255);
    break;
}
if ((joint0State == TrackingState_Tracked) && (joint1State == TrackingState_Tracked))
{
    line(SkeletonImage, pointSet[joint0], pointSet[joint1], color, 2);
}
else
{
    line(SkeletonImage, pointSet[joint0], pointSet[joint1], color, 2);
}
void drawSkeleton(Mat& SkeletonImage, CvPoint pointSet[], const Joint * pJoints, int whichone)
{


DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_Head, JointType\_Neck);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_Neck, JointType\_SpineShoulder);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_SpineShoulder, JointType\_SpineMid);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_SpineMid, JointType\_SpineBase);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_SpineShoulder, JointType\_ShoulderRight);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_SpineShoulder, JointType\_ShoulderLeft);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_SpineBase, JointType\_HipRight);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_SpineBase, JointType\_HipLeft);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_ShoulderRight, JointType\_ElbowRight);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_ElbowRight, JointType\_WristRight);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_WristRight, JointType\_HandRight);



In [ ]:
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_HandRight, JointType_HandTipRight);
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_WristRight, JointType_ThumbRight);


In [ ]:
//Left Arm
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_ShoulderLeft, JointType_ElbowLeft);
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_ElbowLeft, JointType_WristLeft);
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_WristLeft, JointType_HandLeft);
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_HandLeft, JointType_HandTipLeft);
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_WristLeft, JointType_ThumbLeft);


//Right Leg

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_HipRight, JointType\_KneeRight);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_KneeRight, JointType\_AnkleRight);

DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType\_AnkleRight, JointType\_FootRight);



In [ ]:
//Left Leg
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_HipLeft, JointType_KneeLeft);
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_KneeLeft, JointType_AnkleLeft);
DrawBone(SkeletonImage, pointSet, pJoints, whichone, JointType_AnkleLeft, JointType_FootLeft);
}


## (4) 相关的计算如下:


In [ ]:
//计算向量
CvPoint3D32f GetVector(CvPoint3D32f point1, CvPoint3D32f point2)
{
    CvPoint3D32f V;
    V.x = point2.x - point1.x;
    V.y = point2.y - point1.y;
    V.z = point2.z - point1.z;
    return V;
}


In [ ]:
//计算叉积
CvPoint3D32f CrossProduct(CvPoint3D32f V1, CvPoint3D32f V2)
{
    CvPoint3D32f V;
    V.x = V1.y * V2.z - V1.z * V2.y;
    V.y = V1.z * V2.x - V1.x * V2.z;
    V.z = V1.x * V2.y - V1.y * V2.x;
    return V;
}
//计算向量间的夹角
float AngerOf(CvPoint3D32f V1, CvPoint3D32f V2)
{
    float anger;
    anger = acos((V1.x * V2.x + V1.y * V2.y + V1.z * V2.z) / (sqrt(V1.x * V1.x + V1.y * V1.y + V1.z * V1.z) * sqrt(V2.x * V2.x + V2.y * V2.y + V2.z * V2.z)));
    return anger;
}
//函数，计算向量加法
CvPoint3D32f VectorPlus(CvPoint3D32f point1, CvPoint3D32f point2)
{
    CvPoint3D32f point;
    point.x = point2.x + point1.x;
    point.y = point2.y + point1.y;
    point.z = point2.z + point1.z;
    return point;
}
//函数，计算向量与数相乘
CvPoint3D32f VectorMultiply(float k, CvPoint3D32f point1)
{
    CvPoint3D32f point;
    point.x = k * point1.x;
    point.y = k * point1.y;
    point.z = k * point1.z;
    return point;
}


本章介绍了机器人操作系统在 Baxter 机器人上的应用实例，包括了如何使用 ROS 对机器人进行远程控制，如何借助 ROS 在机器人上实现控制算法等课题。

